# Réimplémenter notre Workflow avec l'OpenAI Agents SDK

## 1. Introduction

Au cours des deux sessions précédentes, nous avons exploré et implémenté l'orchestration par **Graphes d'États**. Ce paradigme (popularisé par LangGraph) impose une structure géométrique rigide et centralisée où un orchestrateur contrôle minutieusement chaque transition d'état au sein d'une machine à états compilée.

Aujourd'hui, nous allons étudier une philosophie diamétralement opposée : **l'orchestration décentralisée par transferts de contrôle (Handoffs)**, en utilisant le framework officiel d'OpenAI, l'**OpenAI Agents SDK** (successeur de la bibliothèque de recherche *Swarm*).

Ici, plus de graphes compilés, plus de nœuds ni d'arêtes virtuelles. Les agents collaborent en tant que pairs autonomes. Chaque agent est équipé d'instructions, d'outils et d'une liste d'autres agents vers lesquels il peut choisir de déléguer la conversation s'il estime ne plus être le plus qualifié pour répondre. C'est l'agent lui-même, par le biais d'un appel d'outil de transfert, qui pilote dynamiquement le flux d'exécution.

## 2. Où en sommes-nous?
- Jour 1 : Fondations théoriques et analyse de l'écosystème des frameworks.
- Jour 2 : Codage de notre micro-framework linéaire à partir de zéro.
- Jour 3 : Élaboration de notre propre moteur de graphe cyclique (*StateGraph*).
- Jour 4 : Migration de notre architecture vers **LangGraph**.
- Jour 5 (Aujourd'hui - Ce module) : Migration et réimplémentation du même workflow rédactionnel avec l'**OpenAI Agents SDK**.
- Jour 6 : Comparaison comparative de **CrewAI, Google ADK et AutoGen**.

## 3. Objectifs pédagogiques du Jour 5

À l'issue de cette journée, vous serez capable de :
- Expliquer la différence fondamentale entre l'orchestration géométrique centralisée (graphes) et l'orchestration décentralisée (handoffs).
- Concevoir un système multi-agents en utilisant les quatre primitives clés de l'OpenAI Agents SDK (Agent, handoff, Runner, Context).
- Résoudre le problème des dépendances circulaires lors de la déclaration de transferts bidirectionnels en Python.
- Manipuler des variables d'état partagées et mutables via l'injection du wrapper de contexte RunContextWrapper.
- Optimiser la consommation de jetons (Tokens) en configurant des filtres d'historique de conversation (input_filter).

## 4. Graphes vs Handoffs : Le Duel Philosophique

La conception de systèmes multi-agents en production oppose deux visions de l'ingénierie logicielle  : 
```
        Orchestration par Graphe (LangGraph)
    +----------------------------------------+
    |           StateGraph (Central)         |
    |  -> [Node A] -> ->... |  <-- Contrôle rigide externe
    +----------------------------------------+

       Orchestration par Handoffs (OpenAI SDK)
    +----------+   transfer_to_b   +----------+
    | Agent A  | ----------------> | Agent B  |  <-- Décision décentralisée
    +----------+                   +----------+

```
1. **La Vision Géométrique (LangGraph)** : Le flux est orchestré de l'extérieur. Les nœuds effectuent des calculs sans savoir d'où ils viennent ni où ils vont. C'est le graphe compilé qui gère l'ordonnancement. Ce pattern est exceptionnel pour des processus d'entreprise complexes et hautement réglementés nécessitant un déterminisme total et des validations humaines strictes.
   
2. **La Vision Organique (OpenAI Agents SDK / Swarm)** : Le flux est géré de l'intérieur. Chaque agent possède ses propres instructions et choisit d'appeler l'outil *transfer_to_specialist* s'il détecte que la requête utilisateur sort de son domaine de compétence. C'est une architecture fluide, extrêmement simple à coder, très proche du fonctionnement des équipes humaines.

## 5. Modélisation de l'État de Transition par Handoffs

Soit un ensemble d'agents spécialisés $\mathcal{A} = \{A_1, A_2, \dots, A_n\}$. À chaque étape temporelle $t$, un unique agent détient le contrôle de la conversation, que l'on note $A_{active} \in \mathcal{A}$.

L'état général du système est modélisé par le tuple :

$$S_t = \left( A_{active}, H_t, C_t \right)$$

Où $H_t$ représente l'historique complet des messages de la conversation courante, et $C_t \in \mathcal{C}$ est le dictionnaire de contexte global mutable partagé entre tous les agents.

Lorsque le modèle de langage associé à l'agent actif $A_{active}$ est interrogé, il produit une réponse $R_t \sim \pi_\theta(H_t, C_t)$.

La transition d'état vers l'étape suivante s'articule ainsi :
- Si la réponse implique un outil classique $T$, l'outil est exécuté localement, l'observation est ajoutée à $H_{t+1}$, et $A_{active}$ conserve le contrôle.
- Si la réponse implique un outil de transfert $\text{transfer\_to\_}A_j$, l'infrastructure exécute le transfert de contrôle. Une fonction optionnelle d'interception d'historique (input filter) $f_{filter}$ peut être appliquée pour nettoyer le contexte :

$$H_{t+1} = f_{filter}(H_t)$$

L'agent actif est mis à jour :

$$A_{active} \leftarrow A_j$$

Le système réexécute immédiatement la boucle d'évaluation avec le nouvel agent $A_j$ et l'historique filtré $H_{t+1}$.

## 6. L'Importance des "Input Filters" et du "Nested History"

Dans un système basé sur des handoffs, le nouvel agent reçoit par défaut l'intégralité de l'historique des discussions passées. Cela présente un avantage : le nouvel agent possède tout le contexte.

Mais cela pose deux problèmes majeurs en production  :
1. L'inflation de contexte : Si l'utilisateur effectue 10 transferts successifs, la taille de l'historique explose, augmentant drastiquement les coûts et la latence.

2. La confusion de rôle : L'agent cible peut être perturbé par les instructions systèmes ou les schémas d'outils obsolètes des agents précédents, provoquant des hallucinations d'appels d'outils.

Pour résoudre ce problème, l'OpenAI Agents SDK propose deux outils avancés :
- Les Input Filters (*input_filter*) : Des fonctions Python de filtrage qui interceptent la conversation lors du transfert d'un agent vers un autre, permettant d'élaguer, de reformater ou de supprimer les messages intermédiaires non pertinents.
- Le Nesting d'Historique (*nest_handoff_history=True*) : Cette option compresse automatiquement le transcript des conversations passées sous la forme d'un unique message de résumé (*<CONVERSATION HISTORY>*), préservant ainsi l'espace de contexte du modèle.

## 7. 🧠 Notes d'Architecte

**Pourquoi OpenAI a-t-il créé son Agents SDK?**

En 2024, OpenAI avait publié Swarm à titre purement éducatif pour démontrer la simplicité des patrons de coordination basés sur les handoffs. Face à l'adoption massive de cette philosophie par les développeurs (fatigués de la complexité de LangChain et LangGraph), OpenAI a officialisé cette approche fin 2025 en lançant l'**OpenAI Agents SDK**.

L'objectif d'OpenAI est de fournir un cadre de développement "sans couture" (zero ceremony), où la coordination des agents utilise les mécanismes de bas niveau natifs de l'API des modèles (comme les Responses API et le Function Calling standard), garantissant des performances maximales et une latence minimale.

**Le Problème de la Dépendance Circulaire en Python**

Lorsque deux agents s'appellent mutuellement (le Writer doit pouvoir transférer au Critic, et le Critic doit pouvoir retransférer au Writer), nous faisons face à un problème classique de programmation : **la circularité.**

Si nous écrivons :

```
writer_agent = Agent(name="Writer", handoffs=[critic_agent]) # Erreur : critic_agent n'est pas encore défini!
critic_agent = Agent(name="Critic", handoffs=[writer_agent])
```
Pour résoudre élégamment cette contrainte, nous devons instancier nos agents de manière indépendante sans lister leurs handoffs immédiatement, puis injecter dynamiquement leurs dépendances une fois toutes les variables déclarées :

```
writer_agent = Agent(name="Writer")
critic_agent = Agent(name="Critic")

# Liaison dynamique des handoffs après instanciation
writer_agent.handoffs = [critic_agent]
critic_agent.handoffs = [writer_agent]
```
Cette souplesse d'écriture est l'une des forces majeures de l'implémentation de l'Agents SDK d'OpenAI.

## 8. 📈 Notes Marché

**La facturation à l'usage et l'optimisation économique en 2026**

En production, le coût total de possession (TCO) d'une architecture agentique est dominé par la consommation des tokens d'entrée. C'est ici que l'OpenAI Agents SDK se positionne comme un choix extrêmement rentable face aux frameworks d'entreprises lourds :

|Framework|Surcharge de jetons par tour (Moyenne)|Complexité d'Infrastructure|ROI en Production|
|---|---|---|---|
|OpenAI Agents SDK|Minimale (uniquement les prompts système et les handoffs actifs)|Très faible (client HTTP standard, pas de state-machine complexe) |Maximal pour des cas d'usage conversationnels ou de routage rapide |
|LangGraph|Moyenne à élevée (génération de schémas d'état verbeux et tracking d'historique de graphes) |Élevée (gestion obligatoire de bases de données de checkpointing)| Élevé pour des flux critiques hautement transactionnels|
|Model Context Protocol (MCP)|Très élevée (les schémas d'outils et de serveurs au format standard JSON-RPC sont verbeux)| Moyenne (architecture client-serveur réseau) |Élevé pour l'interopérabilité immédiate de connecteurs tiers |

L'adoption de l'Agents SDK progresse massivement dans les architectures de micro-services légères, où la vitesse d'onboarding d'un nouveau développeur et la lisibilité du code Python priment sur les structures de graphes complexes.

## 9. Code Source de Production (*examples/writer_critic_openai_sdk.py*)

Note importante : Pour utiliser ce script, vous devez installer le SDK d'OpenAI dédié aux agents :
```
Bash
pip install openai-agents
```
(Bien que le package se nomme *openai-agents* sur PyPI, les importations en Python s'effectuent depuis le module *agents*).

## 10. Suite de Validation Automatisée (*tests/test_openai_agent.py*)

Ce fichier pytest assure la validation logique de notre architecture décentralisée, en testant l'intégrité de nos configurations et le câblage dynamique des handoffs sans consommer de tokens API.

Pour exécuter cette suite d'essais unitaire :

```
pytest tests/test_openai_agent.py
```

## 11. Exercices & Défis

**Exercice : Ajout d'un Guardrail de validation de contenu (Input Guardrail)**

- Objectif : L'OpenAI Agents SDK intègre nativement des politiques de validation des entrées/sorties appelées Guardrails.

- Consignes : Ajoutez une fonction d'analyse ou d'interception d'entrée au *writer_agent* à l'aide du paramètre *input_guardrails*. Ce guardrail doit lever une exception ou bloquer l'exécution si la requête de l'utilisateur contient des termes inappropriés ou ne concerne pas un sujet technologique.

**Défi : Mettre en œuvre un Filtre d'Historique personnalisé (Input Filter)**

- Objectif : Configurer un *input_filter* sur le handoff du Critic vers le Writer.

- Consignes : Lorsque le Critic rejette l'article et repasse la main au Writer, écrivez une fonction de filtrage *input_filter* qui supprime l'intégralité du premier brouillon obsolète de l'historique des messages, pour ne transmettre au Writer que les consignes de correction de l'étape courante, préservant ainsi 50 % de l'espace de contexte du modèle.

## 12. Synthèse
- L'orchestration organique de l'OpenAI Agents SDK élimine la lourdeur d'écriture des machines à états au profit d'arbres de décisions décentralisés.   

- La flexibilité syntaxique du framework permet de partager un unique objet de contexte mutable (*ProjectContext*) entre plusieurs entités d'exécution.

- La gestion des handoffs comme outils standards offre une transition naturelle, rapide à implémenter et hautement performante pour les modèles natifs d'OpenAI.   

## 13. Bibliographie

- OpenAI Developer Portal — Agents Python SDK Reference & Examples : (https://github.com/openai/openai-agents-python)

- OpenAI API Docs — Orchestration, Handoffs, and Guardrails Guide : (https://developers.openai.com/api/docs/guides/agents)

- Weave Integration Guides — Tracing and evaluating OpenAI Agents workflows : (https://docs.wandb.ai/weave/guides/integrations/openai_agents)

## 14. Ce qui sera développé demain

Demain (Semaine 4 — Jour 6), nous réaliserons un grand tour d'horizon comparatif de l'écosystème pour clore ce module d'orchestration. Nous mettrons face à face trois géants complémentaires  :   

1. CrewAI (l'orchestration par rôles et backstories d'équipes).   

2. Google ADK (l'intégration native GCP et les flux multimodaux de Gemini).

3. AutoGen / AG2 (les motifs de discussions asynchrones et collaboratifs complexes).   

À l'issue de cette prochaine journée d'étude, vous posséderez une vision panoramique complète et objective de l'ensemble de la stack technologique de développement d'agents en 2026.   

